### Masked layer (de-id)

Reads the **ingestion** tables and writes the **masked** tables. Same shape as `HA_ingestion`, same function names where the logic is the same.

1. Read the mapping sheets
    - `ingestion_mapping.csv`: `cdm_table` / `cdm_column` = the ingestion table and column to read, `standard_name` / `data_type` = the masked column name and type
    - `rule_mapping.csv`: rows with `rule_type = element` and `layer = de-id`. `type_object` = table, `object_name` = ingestion column, `rule = class:<name>` points at a `rule_type = class` row whose `sql_tx` holds the expression (`<element_name>` is replaced by the column)

2. SCD on each table (same split as ingestion)
    - dimension: hashkey based `MERGE`
    - fact: rows land in the masked table, then AUTO CDC pipeline SQL is generated (APPLY CHANGES / SCD type 2)
    - table without a PRIMARY key: append only (warning is logged)

3. Audit and config columns on every table
    - audit: `hashkey, add_date, delete_date, last_modified_date, is_active, is_incremental`
    - config: `data_source_id, data_source, dataset, data_version` all come from the ingestion table. If an ingestion table has no `data_version` column, a warning is logged and `config["masked"]["data_version"]` is used instead

Config (`dev_config.json`), nothing extra is needed:
- `config["ingestion"]["mappath"]` = `ingestion_mapping.csv` (which table and columns to read, `standard_name`, `data_type`)
- `config["masked"]["mappath"]` = the rule sheet (`Rule_mapping.csv`)
- `config["masked"]["data_version"]` is only used when an ingestion table has no `data_version` column

Table names: reads `ingestion_<t>`, writes `masked_<t>`. Facts are append-only (never updated): the table keeps `is_incremental = 1` on the rows it loads, and the view `masked_<t>_v` shows the flag as 1 only for the latest run. Fact pipeline objects are `src_<t>_masked`, `scd_<t>_masked`, `<t>_masked`, so they cannot collide with the ingestion pipeline objects.

In [0]:
%run /Users/pioneerspd@gmail.com/Metadatadriven_proj/HA_control

### HA_control
Reads the weekly load sheet (`Weekly_load_stats.csv`), checks it, and tells a layer notebook which tables to run.

Load it with `%run ./HA_control` as the only line of its own cell. It defines functions only, so it is safe to `%run`.


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import logging
import json, ast, re, os
from datetime import datetime

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    force=True
)
logger = logging.getLogger(__name__)

In [0]:
SRC_PREFIX = "ingestion_"      # tables written by HA_ingestion
TGT_PREFIX = "masked_"         # tables written by this notebook
RULE_LAYER = "de-id"           # rule_mapping.layer handled here
CKPT_BASE  = "/Volumes/dab/healthcare_datasets/healthcare_vol/data/checkpoint"   # one folder per table, never shared with ingestion
VOL_DATA_BASE = "/Volumes/dab/healthcare_datasets/healthcare_vol/data"
OUT_DIR    = "/Workspace/pipelines/sql"

# one timestamp per run (set when this cell runs): every fact row loaded in the run gets the same add_date,
# so "the rows of the latest run" is a plain filter and nothing has to be rewritten afterwards
RUN_TS = datetime.now()
# upper bound of one streaming micro-batch; availableNow splits a big backlog into several batches
STREAM_MAX_BYTES = "100g"

# same as ingestion, plus decimal (the precision comes from the unnamed column next to data_type)
TYPE_MAP = {"varchar": "string", "string": "string", "int": "int",
            "bigint": "bigint", "double": "double", "date": "date",
            "timestamp": "timestamp", "boolean": "boolean", "decimal": "decimal"}

AUDIT_COLS = {"hashkey", "row_hash", "add_date", "delete_date",
              "last_modified_date", "is_active", "is_incremental"}

def get_vol_path(layer, short_name):
    """Volume path for a layer+table, e.g. masked/masked_mx_claim.
    Creates the parent directory so Delta writes don't fail on a missing path."""
    path = f"{VOL_DATA_BASE}/{layer}/{short_name}"
    os.makedirs(path, exist_ok=True)
    return path

def get_data_version(cfg):
    "data_version is set per layer in dev_config.json; fail loudly instead of guessing a value"
    v = str(cfg.get("data_version") or "").strip()
    if not v:
        raise ValueError("dev_config.json has no data_version for this layer")
    return v

def load_config(path):
    """Load JSON config, auto-repairing missing commas between key-value pairs.
    The config file in the shared volume is occasionally re-saved without commas;
    this catches the breakage, repairs it in place, and re-validates."""
    with open(path, 'r') as f:
        raw = f.read()
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        logger.warning(f"{path}: JSON parse error ({e}), attempting auto-repair")
        lines = raw.split('\n')
        fixed_lines = []
        for i, line in enumerate(lines):
            stripped = line.rstrip()
            if (stripped.endswith('"') and not stripped.endswith(',')
                    and not stripped.endswith('{') and not stripped.endswith('[')
                    and not stripped.endswith(':')
                    and i + 1 < len(lines)
                    and lines[i + 1].lstrip().startswith('"')):
                line = stripped + ','
            fixed_lines.append(line)
        fixed = '\n'.join(fixed_lines)
        with open(path, 'w') as f:
            f.write(fixed)
        logger.info(f"{path}: auto-repaired missing commas, re-validating")
        with open(path, 'r') as f:
            return json.load(f)

In [0]:
def read_mapping(map_path):
    df = spark.read.option("header", True).csv(map_path)
    # a blank header cell (the precision column right of data_type) is read as _c<n>
    blanks = [c for c in df.columns if re.fullmatch(r"_c\d+", c)]
    if blanks:
        df = df.withColumnRenamed(blanks[0], "data_precision")
    return df


def parse_inc_arg(text):
    if not text or not text.strip():
        return {}
    return ast.literal_eval(text.strip())


def get_hash_opts(rules):
    # inc_arg appears only on the first row of each table; same parsing as HA_ingestion
    inc = {}
    for r in rules:
        if not r["inc_arg"]:
            continue
        try:
            inc = parse_inc_arg(r["inc_arg"])
        except (ValueError, SyntaxError):
            inc = {}
        if isinstance(inc, str):
            # Spark's CSV reader doubles the quotes: {'hash_key_reqd':Y}
            try:
                inc = ast.literal_eval(inc.replace(":Y}", ":'Y'}"))
            except (ValueError, SyntaxError):
                inc = {}
        if isinstance(inc, dict) and inc:
            break
    if not isinstance(inc, dict):
        inc = {}
    return (inc.get("EXCEPT_KEY_COLUMN_CHECKSUM_LOAD")
            or inc.get("ALL_COLUMN_CHECKSUM_LOAD")
            or {})


def resolve_dtype(row):
    raw = (row.get("data_type") or "string").strip().lower()
    dtype = TYPE_MAP.get(raw, "string")
    if dtype == "int":
        dtype = "bigint"            # same widening as ingestion (10-digit ids overflow INT)
    if dtype == "decimal":
        prec = (row.get("data_precision") or "").strip()
        if not prec:
            raise ValueError(f"{row.get('cdm_column')}: data_type decimal needs a precision such as (22,7)")
        dtype = f"decimal{prec}"
    return dtype

In [0]:
def read_rules(rule_df):
    """Active rules of the de-id layer -> ({class name: sql_tx}, [element rows])."""
    rows = (rule_df
            .filter(col("is_active_map").cast("int") == 1)
            .filter(lower(trim("layer")) == RULE_LAYER)
            .collect())
    kind = lambda r: (r["rule_type"] or "").strip().lower()
    class_sql = {r["type_object"].strip(): r["sql_tx"].strip()
                 for r in rows if kind(r) == "class" and r["sql_tx"]}
    elements = [r for r in rows if kind(r) == "element"]
    return class_sql, elements


def element_rules_for(elements, cdm_table, key_src_cols, known_tables):
    """{ingestion column: [rule rows]} that apply to this table."""
    out = {}
    for r in elements:
        obj = (r["type_object"] or "").strip().lower()
        c   = (r["object_name"] or "").strip()
        if obj == cdm_table.lower():
            pass
        elif obj not in known_tables and obj == c.lower() and c in key_src_cols:
            # e.g. type_object = provider_id for the provider dimension's key
            logger.warning(f"{cdm_table}.{c}: rule type_object '{obj}' is not a table name; "
                           f"applied because {c} is this table's PRIMARY key. "
                           f"Set type_object to '{cdm_table}' in the rule sheet.")
        else:
            continue
        out.setdefault(c, []).append(r)
    return out


def masked_expr_sql(column, rule_rows, class_sql):
    """Chain the class expressions for one column (ordered by rule_order, rule_set)."""
    sql = f"`{column}`"
    for r in sorted(rule_rows, key=lambda r: (int(r["rule_order"] or 0), int(r["rule_set"] or 0))):
        kind, _, name = (r["rule"] or "").partition(":")
        name = name.strip()
        if kind.strip().lower() != "class" or name not in class_sql:
            raise ValueError(f"{column}: unsupported or unknown rule {r['rule']!r}")
        sql = class_sql[name].replace("<element_name>", f"({sql})")
        try:
            expr(sql)
        except Exception as e:
            raise ValueError(f"{column}: rule class '{name}' has invalid sql_tx "
                             f"({type(e).__name__}); fix it in the rule sheet") from e
    return sql

In [0]:
def read_ingestion(table, is_fact):
    # ingestion facts are append-only landing tables -> stream them; dimensions are full reads
    if is_fact:
        return spark.readStream.option("maxBytesPerTrigger", STREAM_MAX_BYTES).table(table)
    return spark.read.table(table)


def build_masked_df(map_df, rule_df, cdm_table, src_catalog, src_schema, data_version="v1"):
    rules = (map_df
             .filter(col("is_active_map").cast("int") == 1)
             .filter(lower(trim("cdm_table")) == cdm_table.lower())
             .collect())
    if not rules:
        raise ValueError(f"No active mapping rows for {cdm_table!r}")

    target_types = sorted({r["target_type"].strip().lower()
        for r in rules
        if r["target_type"] and r["target_type"].strip()})
    src_name = f"{src_catalog}.{src_schema}.{SRC_PREFIX}{cdm_table}"
    src_df = read_ingestion(src_name, "fact" in target_types)

    # masked column names must be unique (two cdm columns can point at one standard_name)
    names = [(r["standard_name"] or "").strip() for r in rules]
    if "" in names:
        raise ValueError(f"{cdm_table}: a mapping row has no standard_name")
    dups = sorted({n for n in names if names.count(n) > 1})
    if dups:
        raise ValueError(f"{cdm_table}: standard_name used more than once: {dups}")

    is_key = lambda r: (r["map_type"] or "").strip().upper() == "PRIMARY"
    known_tables = {x[0].strip().lower() for x in map_df.select("cdm_table").distinct().collect() if x[0]}
    class_sql, elements = read_rules(rule_df)
    by_col = element_rules_for(elements, cdm_table,
                               {r["cdm_column"].strip() for r in rules if is_key(r)}, known_tables)

    exprs, key_cols, non_key_cols, masked = [], [], [], []
    for r in rules:
        d       = r.asDict()
        src_col = d["cdm_column"].strip()
        tgt     = d["standard_name"].strip()
        dtype   = resolve_dtype(d)
        col_expr = col(f"`{src_col}`")

        if src_col in by_col:
            col_expr = expr(masked_expr_sql(src_col, by_col[src_col], class_sql))
            out_type = src_df.select(col_expr).schema[0].dataType
            if isinstance(out_type, StringType) and dtype != "string":
                # a hash is a string; casting it to a number would return NULL
                logger.warning(f"{cdm_table}.{tgt}: masking returns string, so data_type {dtype} is not applied")
                dtype = "string"
            masked.append((src_col, col_expr))

        exprs.append(col_expr.cast(dtype).alias(tgt))
        (key_cols if is_key(r) else non_key_cols).append(tgt)

    # a rule that turns a value into NULL breaks joins silently, so check it on real data
    # (a stream cannot be counted: read up to 10k rows of the same table as a batch instead)
    chk = src_df if not src_df.isStreaming else read_ingestion(src_name, False).limit(10000)
    for src_col, col_expr in masked:
        if chk.filter(col(f"`{src_col}`").isNotNull() & col_expr.isNull()).limit(1).count():
            raise ValueError(f"{cdm_table}.{src_col}: masking rule returned NULL for a non-null value "
                             f"(check the cast inside the rule, e.g. integer on non-numeric or oversized ids)")

    # config column: data_version is taken from the ingestion table (and is not part of the hash)
    if "data_version" not in names and "data_version" in src_df.columns:
        exprs.append(col("data_version"))
    df = src_df.select(*exprs)

    # hashkey is computed on the MASKED values, same recipe as ingestion
    hash_cols = [c for c in non_key_cols if c != "data_version"]
    opts = get_hash_opts(rules)
    if opts.get("hash_key_reqd") == "Y" and hash_cols:
        df = df.withColumn("row_hash", sha2(concat_ws("||",
             *[coalesce(col(c).cast("string"), lit("<null>")) for c in hash_cols]), 256))

    if "data_version" not in df.columns:
        logger.warning(f"{cdm_table}: ingestion table has no data_version column, using config value {data_version!r}")
        df = df.withColumn("data_version", lit(data_version))

    scd = "Y" in {(r["scd"] or "").strip().upper() for r in rules}
    return df, key_cols, scd, target_types

In [0]:
def SCD_UPSERT_logic(df,key,target,layer="masked"):
    "here we will apply the SCD with the upsert logic"
    view = f"ing_{target.replace('.', '_')}"
    df.createOrReplaceTempView(view)

    # columns with special handling in the MERGE
    audit_cols = {"hashkey", "row_hash", "add_date", "delete_date",
                  "last_modified_date", "is_active", "is_incremental"}

    # keys arrive as a list from build_masked_df; accept a single name too
    keys = list(key) if isinstance(key, (list, tuple)) else [key]

    # everything else in the dataframe is copied as-is
    value_cols = [c for c in df.columns if c not in keys and c not in audit_cols]

    if "row_hash" not in df.columns:
        raise ValueError(f"{target}: source dataframe has no row_hash column")

    set_vals = ",\n".join(f"{c} = src.{c}" for c in value_cols)
    ins_cols = ", ".join([*keys, *value_cols])
    ins_vals = ", ".join([f"src.{k}" for k in keys] + [f"src.{c}" for c in value_cols])
    on_expr  = " and ".join(f"tgt.{k} = src.{k}" for k in keys)

    # first run: create the target table with the audit columns
    if not spark.catalog.tableExists(target):
        short_name = target.split('.')[-1]
        vol_path = get_vol_path(layer, short_name)
        init_df = (df
         .withColumnRenamed("row_hash", "hashkey")
         .withColumn("add_date", current_timestamp())
         .withColumn("delete_date", to_timestamp(lit("2099-12-31 00:00:00")))
         .withColumn("last_modified_date", current_timestamp())
         .withColumn("is_active", lit(1))
         .withColumn("is_incremental", lit(1)))
        init_df.write.format("delta").mode("overwrite").saveAsTable(target)
        init_df.write.format("delta").mode("overwrite").save(vol_path)
        logger.info(f"{target}: initial full load (table + volume: {vol_path})")
        return None

    # audit-column semantics:
    #   not matched (new row)              -> fresh audit columns, is_incremental = 1
    #   matched, hash differs (changed row) -> hashkey, add_date, delete_date, last_modified_date
    #                                         all refreshed; is_active = 1; is_incremental = 1
    #   matched, hash identical (unchanged)  -> only is_active = 1 and is_incremental = 0;
    #                                         add_date/delete_date/last_modified_date untouched
    dim_df = spark.sql(f"""
    MERGE INTO {target} as tgt
    using {view} as src
    on {on_expr}
    when matched and tgt.hashkey <> src.row_hash
    then update set 
    {set_vals},
    tgt.hashkey = src.row_hash,
    tgt.add_date = current_timestamp(),
    tgt.delete_date = '2099-12-31 00:00:00',
    tgt.last_modified_date = current_timestamp(),
    tgt.is_active = 1,
    tgt.is_incremental = 1
    when matched and tgt.hashkey = src.row_hash
    then update set 
    tgt.is_active = 1,
    tgt.is_incremental = 0,
    tgt.data_version = src.data_version
    when not matched
    then insert ({ins_cols}, 
    hashkey,
    add_date, 
    delete_date, 
    last_modified_date, 
    is_active, 
    is_incremental)
    VALUES ({ins_vals}, 
    src.row_hash,
    current_timestamp(), 
    '2099-12-31 00:00:00', 
    current_timestamp(), 
    1, 
    1)""")
    # sync volume copy with the current table state after MERGE
    short_name = target.split('.')[-1]
    vol_path = get_vol_path(layer, short_name)
    spark.table(target).write.format("delta").mode("overwrite").save(vol_path)
    logger.info(f"{target}: MERGE done, volume synced to {vol_path}")
    return dim_df

In [0]:

import os

TEMPLATE_PATH = "/Volumes/dab/healthcare_datasets/healthcare_vol/sql/SCD2_fact.sql"
OUT_DIR       = "/Workspace/Users/pioneerspd@gmail.com/Metadatadriven_proj/pipeline/"      # where the pipeline reads its source files

def scd_cdc_logic(df, src, key_cols, name, scd_type=2, seq_col="last_modified_date"):
    """Render the AUTO CDC pipeline SQL for one table. src = landing table (already has the audit columns), name = short pipeline name."""
    special = {"hashkey", "data_version", "add_date", "delete_date",
               "last_modified_date", "is_active", "is_incremental"}          # the template adds these itself
    value_cols = [c for c in df.columns
                  if c not in key_cols and c not in special and c != "row_hash"]
    q = lambda cs: ", ".join(f"`{c}`" for c in cs)
    with open(TEMPLATE_PATH) as f:
        template = f.read()
    sql = template.format(
        src=src, target=name,
        cols=q([*key_cols, *value_cols]),
        keys=q(key_cols),
        seq_col=seq_col,
        scd_type=scd_type,
        track="TRACK HISTORY ON hashkey" if scd_type == 2 else "",
    )
    os.makedirs(OUT_DIR, exist_ok=True)
    out_path = f"{OUT_DIR}/{name}.sql"
    with open(out_path, "w") as f:
        f.write(sql)
    return out_path

In [0]:
def add_audit_columns(df):
    "same audit columns SCD_UPSERT_logic creates, for tables that are not SCD"
    hk = col("row_hash") if "row_hash" in df.columns else lit(None).cast("string")
    return (df.withColumn("hashkey", hk).drop("row_hash")
              .withColumn("add_date", lit(RUN_TS).cast("timestamp"))
              .withColumn("delete_date", to_timestamp(lit("2099-12-31 00:00:00")))
              .withColumn("last_modified_date", lit(RUN_TS).cast("timestamp"))
              .withColumn("is_active", lit(1))
              .withColumn("is_incremental", lit(1)))


def write_stream_both(df, target, cdm_table, vol_path):
    """Single streaming pass: each batch is written to both the UC table and the
    volume Delta path, avoiding a second stream on the same source."""
    def batch_writer(batch_df, batch_id):
        batch_df.write.format("delta").mode("append").saveAsTable(target)
        batch_df.write.format("delta").mode("append").save(vol_path)
    (df.writeStream
       .option("checkpointLocation", f"{CKPT_BASE}/{TGT_PREFIX}{cdm_table}")
       .outputMode("append")
       .trigger(availableNow=True)
       .foreachBatch(batch_writer)
       .start()
       .awaitTermination())


def create_current_view(target):
    """<target>_v: same table, but is_incremental = 1 only for the rows of the latest run.
    The flag is computed when the view is read, so the big table is never updated."""
    spark.sql(f"""
        CREATE OR REPLACE VIEW {target}_v AS
        SELECT * EXCEPT (is_incremental),
               CASE WHEN add_date = (SELECT MAX(add_date) FROM {target}) THEN 1 ELSE 0 END AS is_incremental
        FROM {target}""")


def load_fact(df, target, cdm_table, layer="masked"):
    "audit columns first (every new row: is_active = 1, is_incremental = 1), then append only"
    out = add_audit_columns(df)
    short_name = target.split('.')[-1]
    vol_path = get_vol_path(layer, short_name)
    # If the table exists but is missing audit columns (or they are all NULL from an
    # older run before audit columns were added), drop it (and the checkpoint) so
    # the stream recreates the table with all columns populated.
    if spark.catalog.tableExists(target):
        cols = {f.name for f in spark.table(target).schema.fields}
        stale = "is_incremental" not in cols
        if not stale and "is_incremental" in cols:
            stale = spark.table(target).filter(col("is_incremental").isNotNull()).limit(1).count() == 0
        if stale:
            logger.warning(f"{target}: stale schema (is_incremental missing or all NULL), dropping for recreation")
            spark.sql(f"DROP TABLE IF EXISTS {target}")
            dbutils.fs.rm(f"{CKPT_BASE}/{TGT_PREFIX}{cdm_table}", True)
    if out.isStreaming:
        write_stream_both(out, target, cdm_table, vol_path)
        # If the volume copy has no data (checkpoint already advanced, no new batches),
        # seed it from the UC table so the volume always has the full dataset
        try:
            vol_has_data = spark.read.format("delta").load(vol_path).limit(1).count() > 0
        except Exception:
            vol_has_data = False
        if not vol_has_data and spark.catalog.tableExists(target):
            logger.info(f"{target}: seeding volume copy from UC table (no new stream batches)")
            spark.table(target).write.format("delta").mode("overwrite").save(vol_path)
    else:
        out.write.format("delta").mode("append").saveAsTable(target)
        out.write.format("delta").mode("append").save(vol_path)
    create_current_view(target)
    return out


def create_masked_table(config, cdm_table, layer="masked", src_layer="ingestion"):
    cfg, src = config[layer], config[src_layer]
    map_df  = read_mapping(src["mappath"])      # ingestion_mapping.csv: tables, columns, standard_name, data_type
    rule_df = read_mapping(cfg["mappath"])      # Rule_mapping.csv: the de-id rules

    df, keys, scd, target_types = build_masked_df(
        map_df, rule_df, cdm_table, src["catalog"], src["schema"], get_data_version(cfg))
    if not df.isStreaming and keys:
        df = df.dropDuplicates(keys)      # one row per masked key

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{cfg['catalog']}`.`{cfg['schema']}`")
    target = f"{cfg['catalog']}.{cfg['schema']}.{TGT_PREFIX}{cdm_table}"
    is_dim = bool({"dim", "dimension"} & set(target_types))

    if scd and not keys:
        logger.warning(f"{cdm_table}: scd = Y but no PRIMARY key in the mapping, nothing to match on. "
                       f"Loading append-only with audit columns; mark the business key PRIMARY to get SCD.")
        scd = False

    if scd and is_dim:
        # dimension: hashkey MERGE, runs in this notebook
        SCD_UPSERT_logic(df, keys, target, layer=layer)
    elif scd:
        # fact: the masked table itself carries the audit columns; the pipeline SQL adds the SCD history on top
        out = load_fact(df, target, cdm_table, layer=layer)
        sql_path = scd_cdc_logic(out, target, keys, f"{cdm_table}_masked")
        logger.info(f"{cdm_table}: generated {sql_path}; run the pipeline to build masked_{cdm_table}")
    elif is_dim:
        short_name = target.split('.')[-1]
        vol_path = get_vol_path(layer, short_name)
        dim_df = add_audit_columns(df)
        (dim_df.write.format("delta").mode("overwrite")
           .option("overwriteSchema", "true").saveAsTable(target))
        (dim_df.write.format("delta").mode("overwrite")
           .option("overwriteSchema", "true").save(vol_path))
    else:
        load_fact(df, target, cdm_table, layer=layer)

    logger.info(f"Processed {target}")
    return df

In [0]:
config_path = '/Volumes/dab/healthcare_datasets/healthcare_vol/config/dev_config.json'

dbutils.widgets.text("source", "sphub");  source = dbutils.widgets.get("source")
dbutils.widgets.text("run_id", "manual"); run_id = dbutils.widgets.get("run_id")

config = load_config(config_path)
LOG_TABLE = config["control"]["log_table"]

mapping_rows = (
        read_mapping(config["ingestion"]["mappath"])
          .filter(col("is_active_map").cast("int") == 1)
          .groupBy(trim("cdm_table").alias("cdm_table"))
          .agg(max(lower(trim("target_type"))).alias("target_type"))
          .collect()
          )

known    = {r["cdm_table"].lower() for r in mapping_rows}
run_list = get_run_list(config["control"]["path"], "masked", source, known)
enabled  = {r["table_name"] for r in run_list}

tables = [r for r in mapping_rows if r["cdm_table"].lower() in enabled]
tables = sorted(tables, key=lambda r: 0 if r["target_type"] in ("dim", "dimension") else 1)

failed = []
for r in tables:
    t0 = datetime.now()
    try:
        create_masked_table(config, r["cdm_table"])
        log_run(run_id, "masked", r["cdm_table"], "SUCCESS", t0)
    except Exception as e:
        logger.error(f"{r['cdm_table']} failed: {e}")
        log_run(run_id, "masked", r["cdm_table"], "FAILED", t0, str(e))
        failed.append(r["cdm_table"])

if failed:
    raise RuntimeError(f"Failed tables: {failed}")

2026-10-27 08:47:40,637 | WARNING | __main__ | provider.provider_id: masking returns string, so data_type bigint is not applied
2026-10-27 08:47:51,653 | INFO | __main__ | devproduct.dev.masked_provider: MERGE done, volume synced to /Volumes/dab/healthcare_datasets/healthcare_vol/data/masked/masked_provider
2026-10-27 08:47:51,654 | INFO | __main__ | Processed devproduct.dev.masked_provider
2026-10-27 08:47:57,120 | WARNING | __main__ | patient.patient_id: masking returns string, so data_type bigint is not applied
2026-10-27 08:48:06,418 | INFO | __main__ | devproduct.dev.masked_patient: MERGE done, volume synced to /Volumes/dab/healthcare_datasets/healthcare_vol/data/masked/masked_patient
2026-10-27 08:48:06,420 | INFO | __main__ | Processed devproduct.dev.masked_patient
2026-10-27 08:48:19,650 | INFO | __main__ | devproduct.dev.masked_diagnosis_code: MERGE done, volume synced to /Volumes/dab/healthcare_datasets/healthcare_vol/data/masked/masked_diagnosis_code
2026-10-27 08:48:19,651 